# Módulo 1.4 — Pipeline e ColumnTransformer (prática complementar)

**Disciplina:** Programação em Python para IA — IFNMG/Ceadi

Notebook **complementar** à videoaula: você usa **Pipeline** + **ColumnTransformer** para tratar colunas de tipos diferentes num passo só, **sem data leakage**, e vê uma **demonstração do vazamento**.

**No Colab:** faça upload de `triagem-covid-amostra-raw.csv` (arraste para a barra lateral). Rode as células na ordem.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Carregar e conhecer os dados
Cada linha é um caso de triagem. `target` é o rótulo: 1 = COVID confirmado, 0 = descartado.

In [3]:
df = pd.read_csv("triagem-covid-amostra-raw.csv")
print("linhas, colunas:", df.shape)
df.head()

linhas, colunas: (4000, 13)


,estado,idade,racacor,estacao,sint_tosse,sint_febre,sint_dor_cabeca,sint_dor_garganta,sint_coriza,sint_dispneia,sint_anosmia_ageusia,cond_respiratoria,target
0,RJ,50,Nao informado,verao,0,0,0,0,0,0,0,0,1
1,SP,24,Parda,outono,0,1,0,1,1,0,0,0,0
2,SP,6,Branca,outono,1,1,0,0,1,0,0,0,0
3,SP,6,Branca,inverno,0,0,0,0,0,0,0,0,0
4,RJ,37,Outra,primavera,0,1,1,0,0,0,0,0,0


### Tipos de coluna
Repare que há **tipos diferentes**: texto (categóricas), número inteiro (`idade`) e 0/1 (sintomas).

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   estado                4000 non-null   object
 1   idade                 4000 non-null   int64 
 2   racacor               4000 non-null   object
 3   estacao               4000 non-null   object
 4   sint_tosse            4000 non-null   int64 
 5   sint_febre            4000 non-null   int64 
 6   sint_dor_cabeca       4000 non-null   int64 
 7   sint_dor_garganta     4000 non-null   int64 
 8   sint_coriza           4000 non-null   int64 
 9   sint_dispneia         4000 non-null   int64 
 10  sint_anosmia_ageusia  4000 non-null   int64 
 11  cond_respiratoria     4000 non-null   int64 
 12  target                4000 non-null   int64 
dtypes: int64(10), object(3)
memory usage: 406.4+ KB


### Estatísticas das colunas numéricas
`describe()` resume contagem, média, desvio, mínimo e máximo. Veja a faixa de `idade`.

In [5]:
df.describe()

,idade,sint_tosse,sint_febre,sint_dor_cabeca,sint_dor_garganta,sint_coriza,sint_dispneia,sint_anosmia_ageusia,cond_respiratoria,target
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,30.951000,0.540750,0.367250,0.420250,0.364000,0.451000,0.096750,0.071750,0.018250,0.474000
std,13.814424,0.498399,0.482116,0.493661,0.481209,0.497655,0.295654,0.258106,0.133871,0.499386
min,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,37.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,37.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,50.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 2. Separar X / y e os tipos de coluna
- **Categóricas** (texto): `estado`, `racacor`, `estacao`
- **Numérica contínua**: `idade`
- **Binárias (0/1)**: os sintomas

In [6]:
y = df["target"]
X = df.drop(columns=["target"])

colunas_categoricas = ["estado", "racacor", "estacao"]
colunas_numericas = ["idade"]
print("categóricas:", colunas_categoricas)
print("numérica contínua:", colunas_numericas)
print("total de colunas de entrada:", X.shape[1])

categóricas: ['estado', 'racacor', 'estacao']
numérica contínua: ['idade']
total de colunas de entrada: 12


## 3. Dividir em treino e teste
`train_test_split` separa os dados em dois conjuntos: o modelo **aprende no treino** e é **avaliado no teste** (dados que ele nunca viu).
- `test_size=0.2`: 20% para teste.
- `stratify=y`: mantém a mesma proporção de classes nos dois conjuntos.
- `random_state=42`: torna a divisão reproduzível (sempre a mesma).

In [7]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("treino:", X_tr.shape[0], "| teste:", X_te.shape[0])

treino: 3200 | teste: 800


## 4. Por que Pipeline? (evitar data leakage)
Se você transformar os dados **antes** de separar treino/teste, informação do teste pode "vazar" para o treino.

**Ajustar (`fit`) não é aplicar (`transform`).** O `Pipeline` aprende os parâmetros **só no treino** e depois **aplica também no teste**, com os parâmetros do treino.

Isso importa de verdade na **normalização**: o `StandardScaler` calcula **média e desvio**. Se ele vir o teste, usa estatísticas do teste — e a métrica fica otimista (você verá isso na seção 9). Com `OneHotEncoder` o efeito é menor, porque ele não usa o `y` e as categorias já aparecem no treino.

## 5. ColumnTransformer: cada coluna, seu tratamento
- categóricas → `OneHotEncoder` (viram colunas 0/1)
- `idade` → `StandardScaler` (normaliza: média 0, desvio 1)
- sintomas (0/1) → passam direto (`passthrough`)

In [8]:
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
        ("num", StandardScaler(), colunas_numericas),
    ],
    remainder="passthrough",
)

## 6. Encadear pré-processamento + modelo num Pipeline

In [9]:
modelo = Pipeline(
    steps=[
        ("pre", pre),
        ("clf", MLPClassifier(hidden_layer_sizes=(128, 16), activation="relu",
                               max_iter=300, early_stopping=True, random_state=42)),
    ]
)

## 7. Treinar e avaliar (acurácia)
`modelo.fit` ajusta o pré-processamento **e** treina a MLP — tudo só com o treino.

In [10]:
modelo.fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)
acc = accuracy_score(y_te, y_pred)
print("Acurácia:", round(acc, 3))

Acurácia: 0.594


## 8. Matriz de confusão e classification_report (revisão)
Como no Módulo 1.3: acertos e erros por classe, e precision/recall/F1.

In [11]:
print("Matriz de confusão [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_te, y_pred))
print()
print(classification_report(y_te, y_pred, digits=2))

Matriz de confusão [[TN, FP], [FN, TP]]:
[[269 152]
 [173 206]]

              precision    recall  f1-score   support

           0       0.61      0.64      0.62       421
           1       0.58      0.54      0.56       379

    accuracy                           0.59       800
   macro avg       0.59      0.59      0.59       800
weighted avg       0.59      0.59      0.59       800



## 9. Demonstração do data leakage
Vamos comparar o jeito **correto** (pré-processar só no treino, via Pipeline) com o jeito **errado** (ajustar o pré-processamento na base inteira antes de dividir).

In [12]:
from sklearn.base import clone

# ERRADO: ajustar (fit) o pré-processamento vendo treino + teste
pre_vaza = clone(pre)
X_todo = pre_vaza.fit_transform(X)   # vê a base inteira
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_todo, y, test_size=0.2, random_state=42, stratify=y)
clf2 = MLPClassifier(hidden_layer_sizes=(128, 16), max_iter=300, early_stopping=True, random_state=42)
clf2.fit(Xtr2, ytr2)

print("Acurácia CORRETA (fit só no treino): ", round(acc, 3))
print("Acurácia com VAZAMENTO (fit na base toda):", round(accuracy_score(yte2, clf2.predict(Xte2)), 3))
print("\nA versão com vazamento costuma parecer melhor — é uma estimativa otimista, enganosa.")

Acurácia CORRETA (fit só no treino):  0.594
Acurácia com VAZAMENTO (fit na base toda): 0.626

A versão com vazamento costuma parecer melhor — é uma estimativa otimista, enganosa.


## 10. Quantas colunas saíram do ColumnTransformer?

In [13]:
n_features = modelo.named_steps["pre"].transform(X_tr).shape[1]
print("nº de colunas após o ColumnTransformer:", n_features)

nº de colunas após o ColumnTransformer: 20


## 11. Para pensar
Na seção 9, por que o `StandardScaler` ajustado na base inteira deixa a acurácia otimista?